In [ ]:
# Per-treebank summary table
summary = df.groupby('treebank')[feature_cols].agg(['mean', 'std']).round(3)
summary.columns = [f"{col}_{stat}" for col, stat in summary.columns]
print("Per-treebank feature summary:\n")
print(summary.to_string())

# Print a sample example with all features
print("\n\nSample example (first entry):")
sample = all_examples[0]
inp = json.loads(sample['input'])
out = json.loads(sample['output'])
print(f"  Text: {inp['text'][:100]}...")
print(f"  Language: {sample['metadata_language_code']}")
print(f"  Treebank: {sample['metadata_treebank_id']}")
print(f"  Sentence length: {out['sentence_length']}")
print(f"  Tree depth: {out['tree_depth']}")
print(f"  Mean dependency distance: {out['mean_dd']}")
print(f"  Head-direction entropy: {out['head_direction_entropy']}")
print(f"  Projectivity proportion: {out['projectivity_proportion']}")
print(f"  Functional ratio: {out['functional_ratio']}")
print(f"  Content ratio: {out['content_ratio']}")

print(f"\n--- Demo complete: {len(all_examples)} examples from {len(data['datasets'])} treebanks ---")

## Results Summary

Per-treebank aggregated statistics for all 17 computed features.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle('Syntactic Features Across UD Treebanks', fontsize=14, fontweight='bold')

# 1. Distribution of sentence lengths
ax = axes[0, 0]
for tb in df['treebank'].unique():
    subset = df[df['treebank'] == tb]
    ax.hist(subset['sentence_length'], bins=15, alpha=0.5, label=tb)
ax.set_xlabel('Sentence Length')
ax.set_ylabel('Count')
ax.set_title('Sentence Length Distribution')
ax.legend(fontsize=7)

# 2. Tree depth vs sentence length
ax = axes[0, 1]
for tb in df['treebank'].unique():
    subset = df[df['treebank'] == tb]
    ax.scatter(subset['sentence_length'], subset['tree_depth'], alpha=0.6, s=20, label=tb)
ax.set_xlabel('Sentence Length')
ax.set_ylabel('Tree Depth')
ax.set_title('Tree Depth vs Sentence Length')
ax.legend(fontsize=7)

# 3. Mean dependency distance by treebank
ax = axes[0, 2]
tb_means = df.groupby('treebank')['mean_dd'].mean().sort_values()
ax.barh(tb_means.index, tb_means.values, color='steelblue')
ax.set_xlabel('Mean Dependency Distance')
ax.set_title('Avg Dependency Distance by Treebank')

# 4. Head-direction entropy distribution
ax = axes[1, 0]
for tb in df['treebank'].unique():
    subset = df[df['treebank'] == tb]
    ax.hist(subset['head_direction_entropy'], bins=15, alpha=0.5, label=tb)
ax.set_xlabel('Head-Direction Entropy')
ax.set_ylabel('Count')
ax.set_title('Head-Direction Entropy Distribution')
ax.legend(fontsize=7)

# 5. Projectivity proportion by treebank
ax = axes[1, 1]
tb_proj = df.groupby('treebank')['projectivity_proportion'].mean().sort_values()
ax.barh(tb_proj.index, tb_proj.values, color='coral')
ax.set_xlabel('Mean Projectivity Proportion')
ax.set_title('Projectivity by Treebank')

# 6. Functional vs content ratio scatter
ax = axes[1, 2]
for tb in df['treebank'].unique():
    subset = df[df['treebank'] == tb]
    ax.scatter(subset['functional_ratio'], subset['content_ratio'], alpha=0.6, s=20, label=tb)
ax.plot([0, 1], [1, 0], 'k--', alpha=0.3, linewidth=1)
ax.set_xlabel('Functional Ratio')
ax.set_ylabel('Content Ratio')
ax.set_title('Functional vs Content Ratio')
ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig('feature_plots.png', dpi=100, bbox_inches='tight')
plt.show()
print("Plots saved to feature_plots.png")

## Visualization

Visualize key syntactic features across treebanks and languages.

In [ ]:
# Build a DataFrame from all pre-computed examples
rows = []
for ex in all_examples:
    out = json.loads(ex['output'])
    row = {
        'treebank': ex['metadata_treebank_id'],
        'language': ex['metadata_language_code'],
        'fold': ex['metadata_fold'],
        'sent_id': ex['metadata_sent_id'],
    }
    # Add all scalar output features (skip dd_list)
    for k, v in out.items():
        if k != 'dd_list':
            row[k] = v
    rows.append(row)

df = pd.DataFrame(rows)
print(f"DataFrame shape: {df.shape}")
print(f"\nFeature summary statistics:")
feature_cols = ['sentence_length', 'tree_depth', 'mean_dd', 'dd_variance',
                'projectivity_proportion', 'mean_ic', 'functional_ratio',
                'head_direction_entropy']
df[feature_cols].describe().round(3)

## Parse All Examples into a DataFrame

Extract pre-computed output features into a structured table for analysis and visualization.

In [ ]:
# Flatten all examples from all datasets
all_examples = []
for ds in data['datasets']:
    for ex in ds['examples']:
        all_examples.append(ex)

print(f"Total examples available: {len(all_examples)}")

# Re-compute features on N_EXAMPLES_TO_VERIFY examples
verified = 0
mismatches = 0
for ex in all_examples[:N_EXAMPLES_TO_VERIFY]:
    inp = json.loads(ex['input'])
    expected_out = json.loads(ex['output'])

    computed = compute_sentence_features(
        tokens=inp['tokens'], heads=inp['heads'],
        deprels=inp['deprels'], upos_strs=inp['upos'],
        feats=inp['feats'], text=inp['text'],
    )

    if computed is None:
        print(f"  WARNING: compute returned None for sent_id={ex['metadata_sent_id']}")
        continue

    verified += 1
    # Compare key scalar features (skip dd_list which is a list)
    for key in ['sentence_length', 'tree_depth', 'mean_dd', 'projectivity_proportion', 'head_direction_entropy']:
        if abs(computed[key] - expected_out[key]) > 1e-3:
            print(f"  MISMATCH on {key}: computed={computed[key]}, expected={expected_out[key]}")
            mismatches += 1

print(f"\nVerified {verified}/{N_EXAMPLES_TO_VERIFY} examples, {mismatches} mismatches")
if mismatches == 0:
    print("All features match the pre-computed values!")

## Verify Feature Computation

Re-compute features on a subset of examples and compare against the pre-computed output to verify correctness.

In [ ]:
def upos_int_to_str(upos_int: int) -> str:
    """Convert UPOS integer index to string label."""
    if 0 <= upos_int < len(UPOS_NAMES):
        return UPOS_NAMES[upos_int]
    return "_"


def validate_tree(heads: list[int]) -> bool:
    """Validate tree well-formedness: single root, valid indices, no cycles."""
    n = len(heads)
    if n == 0:
        return False
    roots = [i for i, h in enumerate(heads) if h == 0]
    if len(roots) != 1:
        return False
    for h in heads:
        if h < 0 or h > n:
            return False
    for i in range(n):
        if heads[i] == 0:
            continue
        visited = set()
        cur = i
        while cur not in visited:
            visited.add(cur)
            h = heads[cur]
            if h == 0:
                break
            cur = h - 1
            if cur < 0 or cur >= n:
                return False
        else:
            return False
    return True


def compute_tree_depth_and_arity(heads: list[int]) -> tuple[int, int, float]:
    """Compute tree depth, max arity, and mean arity via BFS from root."""
    n = len(heads)
    children = defaultdict(list)
    root_idx = -1
    for i in range(n):
        if heads[i] == 0:
            root_idx = i
        else:
            children[heads[i] - 1].append(i)
    if root_idx == -1:
        return -1, 0, 0.0
    max_depth = 0
    queue = [(root_idx, 0)]
    while queue:
        node, depth = queue.pop(0)
        if depth > max_depth:
            max_depth = depth
        for child in children[node]:
            queue.append((child, depth + 1))
    non_leaf_arities = [len(ch) for ch in children.values() if len(ch) > 0]
    max_arity = max(non_leaf_arities) if non_leaf_arities else 0
    mean_arity = sum(non_leaf_arities) / len(non_leaf_arities) if non_leaf_arities else 0.0
    return max_depth, max_arity, mean_arity


def get_descendants(heads: list[int]) -> dict[int, set[int]]:
    """Return dict: node (1-indexed) -> set of all descendants (1-indexed)."""
    n = len(heads)
    children = defaultdict(list)
    for i, h in enumerate(heads):
        if h != 0:
            children[h].append(i + 1)
    desc: dict[int, set[int]] = {}
    for node in range(1, n + 1):
        desc[node] = set()
        stack = list(children[node])
        while stack:
            c = stack.pop()
            desc[node].add(c)
            stack.extend(children[c])
    return desc


def is_projective(dep_pos: int, head_pos: int, descendants: dict[int, set[int]]) -> bool:
    """Check if a single dependency arc is projective."""
    a, b = min(dep_pos, head_pos), max(dep_pos, head_pos)
    desc_a = descendants.get(a, set())
    desc_b = descendants.get(b, set())
    for k in range(a + 1, b):
        if k not in desc_a and k not in desc_b:
            return False
    return True


def compute_sentence_features(
    tokens: list[str],
    heads: list[int],
    deprels: list[str],
    upos_strs: list[str],
    feats: list,
    text: str,
) -> dict | None:
    """Compute all 17 features for one sentence. Returns feature dict or None."""
    n = len(tokens)
    effective_length = sum(1 for pos in upos_strs if pos != "PUNCT")
    if effective_length < MIN_EFFECTIVE_LENGTH:
        return None
    if not validate_tree(heads):
        return None
    tree_depth, max_arity, mean_arity = compute_tree_depth_and_arity(heads)
    if tree_depth < 0:
        return None

    descendants = get_descendants(heads)
    head_set = {h for h in heads if h != 0}

    dd_list: list[int] = []
    left_count = 0
    right_count = 0
    proj_count = 0
    ic_list: list[int] = []
    func_count = 0
    total_deps = 0

    for i in range(n):
        pos = i + 1
        h = heads[i]
        if h == 0:
            continue
        total_deps += 1
        dd_list.append(abs(pos - h))
        if h < pos:
            left_count += 1
        else:
            right_count += 1
        if is_projective(pos, h, descendants):
            proj_count += 1
        a, b = min(pos, h), max(pos, h)
        ic_list.append(sum(1 for k in range(a + 1, b) if k in head_set))
        if deprels[i].split(":")[0] in FUNCTIONAL_DEPRELS:
            func_count += 1

    if total_deps == 0:
        return None

    dd_arr = np.array(dd_list, dtype=np.float64)
    mean_dd = round(float(np.mean(dd_arr)), 4)
    dd_variance = round(float(np.var(dd_arr)), 4)
    if len(dd_list) >= 3:
        m, s = np.mean(dd_arr), np.std(dd_arr)
        dd_skewness = round(float(np.mean(((dd_arr - m) / s) ** 3)), 4) if s > 0 else 0.0
    else:
        dd_skewness = 0.0

    ic_arr = np.array(ic_list, dtype=np.float64)
    functional_ratio = round(func_count / total_deps, 4)
    total_dir = left_count + right_count
    if total_dir > 0 and left_count > 0 and right_count > 0:
        p_l, p_r = left_count / total_dir, right_count / total_dir
        hde = round(-(p_l * math.log2(p_l) + p_r * math.log2(p_r)), 4)
    else:
        hde = 0.0

    return {
        "sentence_length": n, "effective_length": effective_length,
        "tree_depth": tree_depth, "max_arity": max_arity,
        "mean_arity": round(mean_arity, 4), "mean_dd": mean_dd,
        "dd_variance": dd_variance, "dd_skewness": dd_skewness,
        "dd_list": dd_list, "projectivity_proportion": round(proj_count / total_deps, 4),
        "n_nonprojective": total_deps - proj_count,
        "mean_ic": round(float(np.mean(ic_arr)), 4),
        "ic_variance": round(float(np.var(ic_arr)), 4),
        "max_ic": int(np.max(ic_arr)),
        "functional_ratio": functional_ratio,
        "content_ratio": round(1.0 - functional_ratio, 4),
        "head_direction_entropy": hde,
    }

print("Feature computation functions defined.")

## Feature Computation Functions

Core functions from the original script for computing syntactic features from dependency trees:
- **Tree validation**: ensures single root, valid indices, no cycles
- **Tree depth & arity**: BFS-based computation
- **Projectivity**: descendant-based arc crossing detection
- **Sentence features**: dependency distance, intervener complexity, head-direction entropy, functional/content ratio

In [ ]:
UPOS_NAMES = [
    "NOUN", "PUNCT", "ADP", "NUM", "SYM", "SCONJ", "ADJ", "PART",
    "DET", "CCONJ", "PROPN", "PRON", "X", "_", "ADV", "INTJ", "VERB", "AUX",
]
FUNCTIONAL_DEPRELS = {"aux", "case", "cc", "clf", "cop", "det", "mark", "punct"}

## Constants

UPOS tag mapping and functional dependency relation categories from the original script.

In [ ]:
# --- Configuration ---
# Number of examples to re-compute features on (for verification)
N_EXAMPLES_TO_VERIFY = 2  # Original: all examples; minimum for demo

# Minimum effective sentence length (non-PUNCT tokens) — matches original script
MIN_EFFECTIVE_LENGTH = 8  # Original: 8

## Configuration

Tunable parameters for the demo. Start with minimum values for quick testing.

In [ ]:
data = load_data()
print(f"Loaded {sum(len(ds['examples']) for ds in data['datasets'])} examples from {len(data['datasets'])} treebanks")
print(f"Languages: {data['metadata']['languages']}")
print(f"Treebanks: {[ds['dataset'] for ds in data['datasets']]}")

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-e4150b-head-directionality-dependent-temporal-d/main/dataset_iter1_universal_depen/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

## Load Data

Load the curated mini demo dataset from GitHub (with local fallback).

In [ ]:
import json
import math
import os
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# All imports used are pre-installed on Colab; install locally to match Colab env
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'matplotlib==3.10.0', 'pandas==2.2.2')

# Universal Dependencies Treebank Feature Dataset

This notebook demonstrates the extraction and analysis of syntactic dependency tree features from Universal Dependencies (UD) treebanks.

**What this dataset does:**
- Processes UD treebanks from `commul/universal_dependencies` on HuggingFace
- Extracts syntactic dependency tree features per sentence (17 computed features)
- Features include: tree depth, dependency distances, projectivity, head-direction entropy, intervener complexity, and more
- Filters sentences with fewer than 8 non-punctuation tokens

This demo loads a curated subset of pre-computed examples and re-computes features to verify correctness, then visualizes key linguistic properties across languages.